# 🗂️ Notebook 2: Collaborative Whiteboard — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/collaborative-whiteboard
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Entities

- **Board** — shared canvas.
- **Shape** — object with id, type, coords.
- **Op** — user edit (add/update/delete).
- **Presence** — user cursor/selection.

## Pydantic models

We use `pydantic` for data validation — it forces us to think about types, required fields, and invariants up front.

In [ ]:
from typing import Literal
from typing import Optional
from pydantic import BaseModel

class Shape(BaseModel):
    id: str
    kind: Literal["rect","circle","line"]
    x: float
    y: float
    w: float = 0
    h: float = 0
    color: str = "#000"

class Op(BaseModel):
    board_id: str
    actor: str
    lamport: int           # logical clock for ordering
    kind: Literal["add","update","delete"]
    shape: Optional[Shape] = None
    shape_id: Optional[str] = None

s = Shape(id="s1", kind="rect", x=10, y=20, w=50, h=30)
op = Op(board_id="b1", actor="alice", lamport=1, kind="add", shape=s)
print(op)

## HTTP APIs

| Method | Path | What |
|---|---|---|
| WS | `/ws/{board_id}` | Subscribe, send ops |
| GET | `/boards/{id}` | Snapshot + since-cursor |
| POST | `/boards` | Create board |


## Quick demo

In [ ]:
# Lamport clock: ensures total-order tie-break across actors
class Lamport:
    def __init__(self): self.t = 0
    def tick(self): self.t += 1; return self.t
    def observe(self, other): self.t = max(self.t, other) + 1; return self.t

a, b = Lamport(), Lamport()
print("a1:", a.tick())       # 1
print("b after seeing a1:", b.observe(1))  # 2
print("b2:", b.tick())       # 3

## Takeaways

- Small, typed models make the service boundary crisp.
- Public APIs hide internal IDs and expose human-friendly resources.
- Write one happy-path test per endpoint before scaling out.